In [ ]:
print("test")

In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
home = '/net/mraid20/export/genie/LabData/Analyses/barakdan/Diet_Microbiome/'

# Analysis

In [ ]:
%cd '/home/barakdan/genie_link/LabData/Analyses/tomerse/diet_mb/figures/reverse'

In [ ]:
import os
print(os.getcwd())  # Prints the current working directory

In [ ]:
import pandas as pd
# from LabData.DataLoaders.GutMBLoader import GutMBLoader
# from LabData.DataLoaders.SubjectLoader import SubjectLoader
# from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
# from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
from sklearn import linear_model
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler
from statistics import mean
from scipy import stats
import matplotlib.pyplot as plt
import pickle
import lightgbm as lgb
import math
import re
from statsmodels.stats.multitest import multipletests

%load_ext autoreload
%autoreload 2

# Predict Dietery Features

In [ ]:
def read_results(df):
    output = []
    for col in df.columns:
        output.append(df[col])
    return tuple(output)

In [ ]:
SPECIES = 'segal_species' # 'segal_species' or 'mpa_species'
species = '' if SPECIES == 'segal_species' else '_mpa'

diet_mb = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features, all_features, targets = loaded_lists

diet_mb

diet_mb.columns = diet_mb.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
all_features = [re.sub(r'[^a-zA-Z0-9_]', '_', x) for x in all_features]
targets = [re.sub(r'[^a-zA-Z0-9_]', '_', x) for x in targets]

diet_mb.head()

In [ ]:
# diet_mb[problems].describe()

In [ ]:
all_features

In [ ]:
all_features

In [ ]:
targets[:5]

In [ ]:
mb_features = targets + ['Richness', 'Shannon_diversity'] + base_features
diet_targets = [feat for feat in all_features if feat not in ['age', 'gender']]
# Exclude "gender" from standardization
features_to_standardize = [feature for feature in mb_features if feature != "gender"]

# Load the scaler and apply it only to the selected features
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/reverse/scaler_mb_diet.pkl', 'rb') as scaler_file:
    scaler = pickle.load(scaler_file)
    diet_mb[features_to_standardize] = pd.DataFrame(
        scaler.transform(diet_mb[features_to_standardize]), 
        columns=diet_mb[features_to_standardize].columns, 
        index=diet_mb[features_to_standardize].index
    )

In [ ]:
diet_mb[diet_targets].describe()

In [ ]:
# diet_mb[mb_features].head()

In [ ]:
# diet_mb[diet_targets].head()

In [ ]:
# diet_targets[:5]

### Read the models results

Base Features

In [ ]:
lgbm_base_scores, lgbm_base_pvalues, lgbm_base_coefs, lgbm_base_preds, lgbm_base_targets = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/reverse/output_base_LGBM_diet.pkl"))
ridge_base_scores, ridge_base_pvalues, ridge_base_coefs, ridge_base_preds, ridge_base_targets = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/reverse/output_base_ridge_diet.pkl"))
print(lgbm_base_scores)
print(ridge_base_scores)

All Features

In [ ]:
lgbm_diet_scores, lgbm_diet_pvalues, lgbm_diet_coefs, lgbm_diet_preds, lgbm_diet_targets = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/reverse/output_LGBM_diet.pkl"))
lgbm_diet_scores = pd.Series(lgbm_diet_scores) 
lgbm_diet_scores.describe()

In [ ]:
# Sorting LGBM diet scores in descending order and selecting the top 10
top_lgbm_scores = lgbm_diet_scores.sort_values(ascending=False).head(10)

# Creating a DataFrame dynamically
df_top_lgbm = pd.DataFrame({
    "Diet Target": [all_features[idx] for idx in top_lgbm_scores.index],  # Map indices to feature names
    "LGBM Score": top_lgbm_scores.values  # Retrieve corresponding scores
})

# Display the DataFrame
print(df_top_lgbm)

In [ ]:
ridge_diet_scores, ridge_diet_pvalues, ridge_diet_coefs, ridge_diet_preds, ridge_diet_targets = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/reverse/output_ridge_diet.pkl"))
ridge_diet_scores = pd.Series(ridge_diet_scores)
ridge_diet_scores.describe()

In [ ]:
# Creating the DataFrame
labeled_scores = pd.DataFrame(
    {"Diet Target": diet_targets, "LGBM Score": lgbm_diet_scores, "LGBM p-value": lgbm_diet_pvalues, 
    "Ridge Score": ridge_diet_scores, "Ridge p-value": ridge_diet_pvalues,                        })

# Display the DataFrame
labeled_scores = labeled_scores.sort_values(by='LGBM Score', ascending=False)

print(labeled_scores.to_string())

In [ ]:
labeled_scores[labeled_scores['Diet Target'] == 'Copper__Cu']

In [ ]:
problematic_features_df = labeled_scores[(labeled_scores['Ridge Score'] > 0.1 ) & (labeled_scores['LGBM Score'] < 0)]
problematic_features = problematic_features_df['Diet Target'].to_list()
print(problematic_features)
print(len(problematic_features))
# problematic_features_df.head(50)

In [ ]:
# diet_mb[problematic_features].describe()

In [ ]:
# problematic_features_df.sort_values(by='Ridge Score', ascending=False)

In [ ]:
top_lgbm = labeled_scores['Diet Target'].head(21).to_list()
print(top_lgbm)
problematic_features += top_lgbm

In [ ]:
print(top_lgbm)
print(len(top_lgbm))

In [ ]:
print(len(problematic_features))
print(problematic_features)

In [ ]:
['Vitamin B-6', 'Plum', 'Valine', 'Arginine', 'Threonine', 'Tyrosine', 'omega_3', 'Fatty acids, total trans', 'Lysine', 'Zinc, Zn', 'Phenylalanine', 'Leucine', 'Methionine', 'Isoleucine', 'Tryptophan', 'Serine', 'Histidine', 'Manganese, Mn', 'Iron, Fe', 'Thiamin', 'Pantothenic acid', 'Cucumber', 'Cystine', 'Aspartic acid', 'Niacin', 'Vitamin B-12', 'Hydroxyproline', 'Fatty acids, total polyunsaturated', 'Vitamin D (D2 + D3)', 'Fatty acids, total saturated', 'Glycine', 'Proline', 'Lettuce', 'Alanine', 'Riboflavin', 'Glutamic acid', 'Copper, Cu', 'omega_6', 'Total lipid (fat)_std', 'wfpb_score_per_day', 'NOVA_food_score', 'med_score_per_day', 'vegan_score_per_day', 'pct_carb_calories', 'paleo_score_per_day', 'plant_protein_pct', 'plant_energy_pct', 'Nutsseedsandproducts', 'Potassium, K', 'pescatarian_score_per_day', 'carnivore_score_per_day', 'Fruits', 'vegetarian_score_per_day', 'pct_fat_calories', 'plant_fat_pct', 'Fiber, total dietary', 'Energy', 'sat_to_total_lipids_ratio', 'Magnesium, Mg', 'Cholesterol']

In [ ]:
import matplotlib.pyplot as plt

# Define number of rows and columns for subplots
num_features = len(problematic_features)
rows = int(num_features**0.5) + 1  # Approximate a square layout
cols = (num_features // rows) + (num_features % rows > 0)  # Adjust columns

fig, axes = plt.subplots(rows, cols, figsize=(15, 15))  # Adjust figure size

# Flatten axes array if needed
axes = axes.flatten()

# Plot histograms for each problematic feature
for i, feature in enumerate(problematic_features):
    diet_mb[feature].hist(ax=axes[i], bins=30, alpha=0.75)
    axes[i].set_title(feature)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()  # Adjust layout for readability
plt.show()


In [ ]:
# Sorting LGBM diet scores in descending order and selecting the top 10
top_ridge_scores = ridge_diet_scores.sort_values(ascending=False).head(10)

# Creating a DataFrame dynamically
df_top_ridge = pd.DataFrame({
    "Diet Target": [all_features[idx] for idx in top_ridge_scores.index],  # Map indices to feature names
    "Ridge Score": top_ridge_scores.values  # Retrieve corresponding scores
})

# Display the DataFrame
print(df_top_ridge)

### Choose Significant Targets without permutations

Second Layer of Filtering, from the distribution of non-significant scores

In [ ]:
def analyze_significance(scores, pvalues, targets, model_name, save_path):
    # Step 1: Apply FDR correction (Benjamini-Hochberg)
    pvalues_adj = multipletests(pvalues, method='fdr_bh', alpha=0.05)[1]
    pvalues_adj = pd.Series(pvalues_adj, index=pvalues.index)
    
    # Step 2: Count significant predictions before and after adjustment
    significants_raw = pvalues[pvalues < 0.05].shape[0]
    significants_adj = pvalues_adj[pvalues_adj < 0.05].shape[0]
    print(f"{significants_raw} significant predictions ({round((significants_raw/len(pvalues))*100, 2)}%)")
    print(f"{significants_adj} significant predictions after adjustment ({round((significants_adj/len(pvalues_adj))*100, 2)}%)")
    
    # Step 3: Extract significant targets
    significant_targets = [target for target, p_adj in zip(targets, pvalues_adj) if p_adj < 0.05]
    print(f"Total targets: {len(targets)}")
    print(f"Significant targets: {len(significant_targets)}")
    
    # Save significant targets
    with open(save_path + f'significant_{model_name}_diet_targets.pkl', 'wb') as file:
        pickle.dump(significant_targets, file)
    
    # Step 4: Create DataFrame
    matched_data = pd.DataFrame({'scores': scores, 'pvalues_adj': pvalues_adj}).dropna()
    
    # Step 5: Compute 99th percentile of non-significant scores
    non_significant_scores = matched_data.loc[matched_data['pvalues_adj'] >= 0.05, 'scores']
    percentile_99 = np.percentile(non_significant_scores, 99)
    print(f"99th percentile of non-significant scores: {percentile_99}")
    
    # Step 6: Apply second filtering step based on the 99th percentile
    matched_data['passes_99th_percentile'] = matched_data['scores'].abs() > percentile_99
    final_matched_data = matched_data[matched_data['pvalues_adj'] < 0.05]
    final_matched_data = final_matched_data[final_matched_data['passes_99th_percentile']]
    
    # Final list of significant targets after both filtering steps
    final_significant_targets = [target for target, score in zip(targets, scores) if score in final_matched_data['scores'].values]
    
    # Step 7: Plot Histogram
    plt.figure(figsize=(8, 5))
    plt.hist(non_significant_scores, bins=20, edgecolor='black', alpha=0.7)
    plt.axvline(percentile_99, color='red', linestyle='dashed', linewidth=2, 
                 label=f'99th Percentile: {percentile_99:.4f}')
    plt.xlabel("Non-Significant Scores")
    plt.ylabel("Frequency")
    plt.title(f"Distribution of Non-Significant Scores with 99th Percentile Cutoff ({model_name})")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return percentile_99, final_significant_targets, final_matched_data, 

### Save significant diet targets (by pearson p-values)

In [ ]:
# LGBM analysis
lgbm_percentile_99, final_lgbm_targets, final_lgbm_data = analyze_significance(
    scores=lgbm_diet_scores, 
    pvalues=lgbm_diet_pvalues, 
    targets=diet_targets, 
    model_name="LGBM", 
    save_path=home_path + "data/reverse/"
)

# Ridge analysis
ridge_percentile_99, final_ridge_targets, final_ridge_data = analyze_significance(
    scores=ridge_diet_scores, 
    pvalues=ridge_diet_pvalues, 
    targets=diet_targets, 
    model_name="ridge", 
    save_path=home_path + "data/reverse/"
)

In [ ]:
# Improvement from linear to LGBM
delta = pd.Series(lgbm_diet_scores) - pd.Series(ridge_diet_scores)
delta.describe()

In [ ]:
print("Improvement median:", delta.median())

In [ ]:
print("Number of dietery features with better LGBM prediction: ", delta[delta > 0].shape[0])
print("Number of dietery features with better Linear prediction: ", delta[delta < 0].shape[0])

## Permutations
In order to find out the size of a significant correlation.

In [ ]:
# # lgbm_div_perm_scores, lgbm_div_perm_pvalues = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/test/reverse/lightGBM_output_diversity_perm.pkl"))
# lgbm_perm = pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/test/reverse/output_LGBM_abundance_perm.pkl")
# diet_perm_scores = lgbm_perm.loc[:, 0]
# # microbes_perm_pvalues = lgbm_mircrobes_perm.loc[:, 1]

In [ ]:
# permutation_scores = diet_perm_scores.iloc[:, :-4]
# percentile_99 = np.percentile(permutation_scores.values.flatten(), 99)
# percentile_99

In [ ]:
# def calc_p_value(perm_scores, true_score):
#     bigger_than_true = (perm_scores >= true_score).sum()
#     print(bigger_than_true)
#     return float(bigger_than_true) / len(perm_scores)

In [ ]:
# top_diet = lgbm_diet_scores.sort_values(ascending=False)
# top_diet = top_diet.head(16)
# top_diet

In [ ]:
# diet_perm_scores

In [ ]:
# mb_names = pd.read_pickle("/home/tomerse/PycharmProjects/pythonProject/data/mb_names.pkl")
# mb_names

In [ ]:
# mb_names_num = mb_names.reset_index()
# mb_names_num[mb_names_num['Unnamed: 0'].isin(["Rep_2746"])]

In [ ]:
# # p value cutoff for p < 0.01
# diet_perm_scores["true_scores"] = lgbm_diet_scores
# diet_perm_scores['true_is_max'] = diet_perm_scores.apply(lambda row: row['true_scores'] >= max(row.drop('true_scores')), axis=1)
# diet_perm_scores['true_is_max'].sum()

In [ ]:
# significant_targets = [target for target, m in zip(targets, diet_perm_scores['true_is_max']) if m]
# print(targets)
# print(significant_targets)


In [ ]:
# significant_targets_index = [index for index, (target, m) in enumerate(zip(targets, diet_perm_scores['true_is_max'])) if m]
# print(targets)
# print(significant_targets_index)


In [ ]:
# with open(home_path + 'data/reverse/significant_targets.pkl', 'wb') as file:
#     pickle.dump(significant_targets, file)

In [ ]:
# print(f"{len(significant_targets)} out of {len(targets)} LGBM predictions ({round((len(significant_targets)/len(targets))*100, 2)}) are significant with p value < 0.01")

In [ ]:
# diet_perm_scores['max'] = diet_perm_scores.drop("true_is_max", axis=1).max(axis=1)
# random_max = diet_perm_scores[~diet_perm_scores['true_is_max']]
# print("Highest overall random score:", random_max['max'].max())

In [ ]:
# diet_perm_scores

In [ ]:
# # Function to count correlations greater or equal to the true correlation
# def count_better_correlations(row):
#     true_corr = row['true_scores']
#     return sum(row[:-1] >= true_corr)

# diet_perm_scores['better_perm_count'] = diet_perm_scores.drop(["true_is_max", "max"], axis=1).apply(count_better_correlations, axis=1)

In [ ]:
# diet_perm_scores['better_perm_count']

In [ ]:
# diet_perm_scores['better_perm_count'][diet_perm_scores['better_perm_count'] > 0]

#### Analyzing non-significant microbes

In [ ]:
# non_significant_microbes = [target for target, m in zip(diet_targets, diet_perm_scores['true_is_max']) if not m]
# non_significant_microbes

In [ ]:
# mb_names.loc[non_significant_microbes].to_csv("non_significant_microbes.csv", index=False)

In [ ]:
# mb_names.loc[non_significant_microbes]

In [ ]:
# mb_names['phylum'].value_counts()

# --- End of permutations section

## Plots

#### Overall results for Eran

Pearson correlations distribution histograms for LGBM & Ridge

We plot here all correlations, these that didn't pass FDR

In [ ]:
def combine_results(scores, pvalues, percentile_99, title):

    df = pd.DataFrame({
        'pearson_correlations': scores,
        'p_values': pvalues
    })
    
    df["is_significant"] = df["p_values"] < 0.05

    # df["p_values"] = df["p_values"].apply(lambda x: f"{x:.50e}")

    df.sort_values(by='pearson_correlations', ascending=False, inplace=True)

    plt.figure(figsize=(10, 5), dpi=300)
    plt.hist(df['pearson_correlations'], bins=20, edgecolor='black', color='cadetblue')
    plt.xlabel("Pearson's Correlation", fontsize=10)
    plt.ylabel("Frequency")

    # Add permutation line later.
    plt.axvline(percentile_99, color='red', linestyle='dotted', linewidth=2, label="99th percentile of non-significants")

    # plt.title('Distribution of Correlation Values Across All Species', fontsize=12)
    plt.legend()

    # Save figure later
    plt.savefig(home + "figures/reverse/ridge_corr_distribution.pdf", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')    
    # plt.savefig(home_path + "figures/reverse/LGBM_corr_distribution.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')   
     
    plt.title(title, fontsize=12)
    plt.show()

    return df

# combined_ridge_results = combine_results(ridge_diet_scores, ridge_diet_pvalues, title="Ridge Correlation Distribution")
# combined_lgbm_results = combine_results(lgbm_diet_scores, lgbm_diet_pvalues, lgbm_percentile_99, title="LGBM Pearson Correlations Distribution")
combined_ridge_results = combine_results(ridge_diet_scores, ridge_diet_pvalues, ridge_percentile_99, title="Ridge Pearson Correlations Distribution")

# print(f"LGBM \n{combined_lgbm_results.head()}\n")
print(f"Ridge \n{combined_ridge_results.head()}")

Need to rearrange the moving average logic (x axis is ?)

In [ ]:
# def moving_average(targets, preds, name):
#     # Sorting by measured values to determine the order
#     order_based_on_measured = np.argsort(targets)

#     # Sorting predicted values based on the order determined by measured values
#     sorted_predicted_values = pd.Series(preds)[order_based_on_measured]
#     # Window sizes
#     window_sizes = [100, 500, 1000]

#     # Plotting moving averages for sorted predicted values
#     plt.figure(figsize=(10, 6))
#     for window_size in window_sizes:
#         moving_avg = pd.Series(sorted_predicted_values).rolling(window=window_size).mean()
#         plt.plot(moving_avg.reset_index(drop=True), label=f'Predicted Values MA-{window_size}', linewidth=1, alpha=0.8)

#     # plt.title(f'Moving Window Average of Predicted Values - {name}')
#     plt.title(f'{name}', fontsize=18)
#     plt.xlabel('Subject Index (Sorted by Measured Values)', fontsize=14)
#     plt.ylabel('Moving Average of Predicted Values', fontsize=14)
#     plt.legend()
#     plt.savefig(home_path + f"figures/reverse/moving_average_{name}.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
#     plt.show()

# moving_average(lgbm_diet_targets[mic_index_list[0]], lgbm_diet_preds[mic_index_list[0]], microbe_name_list[0] + f" - {round(lgbm_diet_scores[mic_index_list[0]], 2)} Correlation")

In [ ]:
# moving_average(lgbm_diet_targets[mic_index_list[1]], lgbm_diet_preds[mic_index_list[1]], microbe_name_list[1] + f" - {round(lgbm_diet_scores[mic_index_list[1]], 2)} Correlation")

In [ ]:
# lgbm_diet_scores.min()

In [ ]:
# moving_average(lgbm_diet_targets[0], lgbm_diet_preds[0], f"Control Species - {round(lgbm_diet_scores[0], 2)} Correlation")

In [ ]:
# lgbm_div_targets

In [ ]:
# moving_average(lgbm_div_targets[0], lgbm_div_preds[0], f"Richness ({round(lgbm_div_scores[0], 2)} Correlation)")

In [ ]:
# moving_average(lgbm_div_targets[1], lgbm_div_preds[1], f"Shannon Diversity ({round(lgbm_div_scores[1], 2)} Correlation)")

In [ ]:
# series_list = []
# microbe_name_list = []
# mic_index_list = []

# for microbe_index in top_diet.head(2).index:
#     mic_index_list.append(microbe_index)
#     mic_perm_scores = permutation_scores.loc[microbe_index]
#     true_score = lgbm_diet_scores.loc[microbe_index]
#     mic_scores = pd.concat([mic_perm_scores, pd.Series([true_score])], ignore_index=True)
#     series_list.append(mic_scores)
#     microbe_name_list.append(mb_names.loc[targets[microbe_index], 'species'])

# print(mic_index_list)
# # Plotting
# # plt.style.use('seaborn-dark')
# # fig, axs = plt.subplots(2, 2, figsize=(10, 10), gridspec_kw={'height_ratios': [1, 1]})
# # fig.delaxes(axs[0][1])

# fig = plt.figure(figsize=(12, 10))
# grid = plt.GridSpec(2, 2, hspace=0.4, wspace=0.3, height_ratios=[1, 1])

# # Histogram on the top spanning two columns
# ax_hist = fig.add_subplot(grid[0, :])
# ax_hist = fig.add_subplot(2, 2, (1, 2))
# ax_hist.hist(lgbm_diet_scores, bins=20, edgecolor='black')
# ax_hist.set_xlabel("Pearson's Correlation")
# ax_hist.set_ylabel("Frequency")
# ax_hist.axvline(percentile_99, color='red', linestyle='dotted', linewidth=2, label="99th percentile of permutations")
# ax_hist.set_title('Distribution of Correlation Values Across All Species')
# ax_hist.set_yticks([])
# ax_hist.set_yticks([])
# ax_hist.legend()
# ax_hist.yaxis.get_major_locator().set_params(nbins=6)

# ax_dot1 = fig.add_subplot(grid[1, 0])
# ax_dot1.scatter(x=lgbm_diet_targets[mic_index_list[0]], y=lgbm_diet_preds[mic_index_list[0]])
# ax_dot1.plot([-4, -1], [-4, -1], 'r--')
# ax_dot1.set_xticks(range(-4, 0))
# ax_dot1.set_yticks(range(-4, 0))
# ax_dot1.set_xlabel("Measured log-abundances")
# ax_dot1.set_ylabel("Predicted log-abundances")
# ax_dot1.set_title(microbe_name_list[0], fontsize=11)

# ax_dot2 = fig.add_subplot(grid[1, 1])
# ax_dot2.scatter(x=lgbm_diet_targets[mic_index_list[1]], y=lgbm_diet_preds[mic_index_list[1]])
# ax_dot2.plot([-4, -1], [-4, -1], 'r--')
# ax_dot2.set_xticks(range(-4, 0))
# ax_dot2.set_yticks(range(-4, 0))
# ax_dot2.set_xlabel("Measured log-abundances")
# ax_dot2.set_ylabel("Predicted log-abundances")
# ax_dot2.set_title(microbe_name_list[1], fontsize=11)

# plt.tight_layout()
# plt.show()

# # plt.style.use('default')

#### How do Ridge and LGBM scores compare?

In [ ]:
%matplotlib inline

to_boxplot = pd.concat([ridge_diet_scores, lgbm_diet_scores] , axis=1)
to_boxplot.columns = ["Ridge", "LGBM"]
# to_boxplot.boxplot(flierprops=dict(markersize=2))
to_boxplot.plot.kde()
plt.title("Pearson Score distributions Ridge VS LGBM")
# plt.ylabel("Pearson Corr")
plt.xticks(rotation=45)
plt.grid()
plt.savefig(home+ "figures/reverse/ridge_vs_lgbm_dist.pdf", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
# plt.savefig(home_path + "figures/reverse/ridge_vs_lgbm_dist.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

In [ ]:
%matplotlib inline
U, p = stats.mannwhitneyu(ridge_diet_scores, lgbm_diet_scores, alternative='two-sided')

plt.rcParams['figure.dpi'] = 300

to_plot = pd.concat([ridge_diet_scores, lgbm_diet_scores] , axis=1)
to_plot.columns = ["Ridge", "LGBM"]
to_plot.plot.scatter("Ridge", "LGBM", alpha=0.5, color='cadetblue', edgecolors='none')
plt.title("Pearson Corr LGBM Vs Ridge")
plt.plot([0, 0.7], [0, 0.7], 'r--')
# plt.grid()
# plt.text(0.45, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home+ "figures/reverse/ridge_vs_lgbm_scatter.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

In [ ]:
# Ridge VS LightGBM
U, p = stats.mannwhitneyu(ridge_diet_scores, lgbm_diet_scores, alternative='two-sided')
to_boxplot = pd.concat([ridge_diet_scores, lgbm_diet_scores] , axis=1)
to_boxplot.columns = ["Ridge", "LightGBM"]
to_boxplot.boxplot(flierprops=dict(markersize=2))
plt.title("Pearson Corr Ridge Vs LightGBM")
plt.ylabel("Pearson Corr")
# plt.xticks(rotation=45)
plt.grid()
plt.text(1.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.3f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home + "figures/reverse/ridge_vs_lgbm_box.pdf", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

### How do age and gender compare?

#### Diet adherence scores targets

In [ ]:
# lgbm_diet_scores = pd.Series(lgbm_diet_scores)
# lgbm_base_scores = pd.Series(lgbm_base_scores)

# to_plot = pd.concat([lgbm_base_scores, lgbm_diet_scores] , axis=1)
# to_plot.columns = ["Age + Gender", "All features"]


# #TODO Change to diet adherence scores
# to_plot.index = ["Richness", "Shannon Index"]
# print(to_plot)
# ax = to_plot.plot(kind="bar", legend=True, figsize=(12, 8), alpha=1, color=["cadetblue", "coral"])
# # plt.title("Diversity Targets")
# plt.ylabel("Pearson Corr")
# plt.xticks(rotation=0)
# plt.legend(bbox_to_anchor=(0.9, 1.0), loc='upper left')


# # Calculate bar positions
# bar_positions = range(len(to_plot))  # x-positions for the bars (0 for Richness, 1 for Shannon)

# # Add separate horizontal lines for each index
# # percentile_99_richness = np.percentile(richness_perm_scores.values.flatten(), 99)
# # percentile_99_shannon = np.percentile(shannon_perm_scores.values.flatten(), 99)

# # Horizontal line for Richness
# # ax.plot(
# #     [bar_positions[0] - 0.4, bar_positions[0] + 0.4],  # x-range covering the Richness bar
# #     [percentile_99_richness, percentile_99_richness],  # y-values for the horizontal line
# #     color='black', linestyle='--', linewidth=1, label="99th percentile (Richness)"
# # )

# # Horizontal line for Shannon Index
# # ax.plot(
# #     [bar_positions[1] - 0.4, bar_positions[1] + 0.4],  # x-range covering the Shannon Index bar
# #     [percentile_99_shannon, percentile_99_shannon],  # y-values for the horizontal line
# #     color='black', linestyle='--', linewidth=1, label="99th percentile (Shannon)"
# # )

# # # Annotate the lines
# # plt.text(bar_positions[0], percentile_99_richness + 0.01, f"{percentile_99_richness:.2f}", color='black', fontsize=10, ha='center')
# # plt.text(bar_positions[1], percentile_99_shannon + 0.01, f"{percentile_99_shannon:.2f}", color='black', fontsize=10, ha='center')

# # to_boxplot.plot.box()
# # plt.savefig(home_path + "figures/reverse/diversity_corr_lgbm_base_diet.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
# plt.show()
# plt.clf()

#### Abundance targets

In [ ]:
U, p = stats.mannwhitneyu(lgbm_base_scores, lgbm_diet_scores, alternative='two-sided')

plt.rcParams['figure.dpi'] = 300
to_plot = pd.concat([lgbm_base_scores, lgbm_diet_scores] , axis=1)
to_plot.columns = ["Age + Gender", "All Features"]
to_plot.plot.scatter("Age + Gender", "All Features", c='cadetblue', edgecolors='none', alpha=0.5)
plt.title("LGBM Pearson Corr base features Vs Full Model")
plt.plot([0, 0.5], [0, 0.5], 'r--')
# plt.grid()
# plt.text(0.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home + "figures/reverse/lgbm_all_vs_base_scatter.pdf", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()


In [ ]:
U, p = stats.mannwhitneyu(lgbm_base_scores, lgbm_diet_scores, alternative='two-sided')
to_boxplot = pd.concat([lgbm_base_scores, lgbm_diet_scores] , axis=1)
to_boxplot.columns = ["Age + Gender", "All Features"]
to_boxplot.boxplot(flierprops=dict(markersize=2))
plt.ylabel("Pearson Corr")
plt.title("LGBM Pearson Corr base features Vs Full Model")
# plt.xticks(rotation=45)
plt.grid()
plt.text(1.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.3f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home_path + "figures/reverse/lgbm_all_vs_base_box.pdf", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

In [ ]:
U, p = stats.mannwhitneyu(ridge_base_scores, ridge_diet_scores, alternative='two-sided')

plt.rcParams['figure.dpi'] = 300
to_plot = pd.concat([ridge_base_scores, ridge_diet_scores] , axis=1)
to_plot.columns = ["Age + Gender", "All Features"]
to_plot.plot.scatter("Age + Gender", "All Features", c='cadetblue', edgecolors='none', alpha=0.5)
plt.title("Ridge Pearson Corr base features Vs Full Model")
plt.plot([0, 0.5], [0, 0.5], 'r--')
# plt.grid()
# plt.text(0.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home_path + "figures/reverse/ridge_all_vs_base_scatter.pdf", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()


In [ ]:
U, p = stats.mannwhitneyu(ridge_base_scores, ridge_diet_scores, alternative='two-sided')
to_boxplot = pd.concat([ridge_base_scores, ridge_diet_scores] , axis=1)
to_boxplot.columns = ["Age + Gender", "All Features"]
to_boxplot.boxplot(flierprops=dict(markersize=2))
plt.ylabel("Pearson Corr")
plt.title("Ridge Pearson Corr base features Vs Full Model")
# plt.xticks(rotation=45)
plt.grid()
plt.text(1.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.3f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home_path + "figures/reverse/ridge_all_vs_base_box.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

#### Comparison between previous features and new features (150 nutrients and dietary patterns)

In [ ]:
prev_diet_score, prev_diet_pvalues, prev_diet_coefs, prev_diet_fnames, prev_diet_preds, prev_diet_targets = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/test/reverse/lightGBM_output_microbes_v2.pkl"))
prev_diet_score = pd.Series(prev_diet_score)

U, p = stats.mannwhitneyu(prev_diet_score, lgbm_diet_scores, alternative='two-sided')
to_plot = pd.concat([prev_diet_score, lgbm_diet_scores] , axis=1)
to_plot.columns = ["Previous", "New"]
to_plot.plot.scatter("Previous", "New", alpha=0.5, color='cadetblue')
plt.title("Pearson Correlation Previous VS New Features (150 nutrients and dietary patterns)")
plt.plot([0, 1], [0, 1], 'r--')
# plt.grid()
plt.text(0.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig("/home/tomerse/PycharmProjects/pythonProject/figures/reverse/prev_vs_new_features.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

Improvement from adding variance features and macros

In [ ]:
# Improvement from adding variance features and macros
lgbm_diet_scores_no_var_features, _, _, _, _  = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/test/reverse/output_LGBM_abundance_no_var_features.pkl"))
delta = pd.Series(lgbm_diet_scores) - pd.Series(lgbm_diet_scores_no_var_features)
delta.describe()

In [ ]:

lgbm_diet_scores_no_var_features = pd.Series(lgbm_diet_scores_no_var_features)

U, p = stats.mannwhitneyu(lgbm_diet_scores_no_var_features, lgbm_diet_scores, alternative='two-sided')
to_plot = pd.concat([lgbm_diet_scores_no_var_features, lgbm_diet_scores] , axis=1)
to_plot.columns = ["Previous", "New"]
to_plot.plot.scatter("Previous", "New", alpha=0.5, color='cadetblue')
plt.title("Pearson Correlation Previous VS New Features (var features and macros)")
plt.plot([0, 1], [0, 1], 'r--')
# plt.grid()
plt.text(0.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
# plt.savefig("/home/tomerse/PycharmProjects/pythonProject/figures/reverse/", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

In [ ]:
# Improvement from adding variance features and macros
lgbm_diet_scores_div_no_var_features, _, _, _, _  = read_results(pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/test/reverse/output_LGBM_div_no_var_features.pkl"))
delta = pd.Series(lgbm_div_scores) - pd.Series(lgbm_diet_scores_div_no_var_features)
delta.describe()

In [ ]:

lgbm_diet_scores_div_no_var_features = pd.Series(lgbm_diet_scores_div_no_var_features)

U, p = stats.mannwhitneyu(lgbm_diet_scores_div_no_var_features, lgbm_div_scores, alternative='two-sided')
to_plot = pd.concat([lgbm_diet_scores_div_no_var_features, lgbm_div_scores] , axis=1)
to_plot.columns = ["Previous", "New"]
to_plot.plot.scatter("Previous", "New", alpha=0.5, color='cadetblue')
plt.title("Diversity Pearson Correlation Previous VS New Features (var features and macros)")
plt.plot([0, 1], [0, 1], 'r--')
# plt.grid()
plt.text(0.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
# plt.savefig("/home/tomerse/PycharmProjects/pythonProject/figures/reverse/", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

#### What is the corr diference between the highly predicted microbes in LGBM and Ridge?
Maybe those with high signal will show a significant difference between the models.

In [ ]:
highly_pred_microbes_lgbm = lgbm_diet_scores.sort_values(ascending=False)
highly_pred_microbes_lgbm = highly_pred_microbes_lgbm[highly_pred_microbes_lgbm > 0.2]
highly_pred_microbes_lgbm

In [ ]:
highly_pred_microbes_ridge = ridge_diet_scores.sort_values(ascending=False)
highly_pred_microbes_ridge = highly_pred_microbes_ridge[highly_pred_microbes_ridge > 0.2]
highly_pred_microbes_ridge

In [ ]:
# Top predictions Ridge vs LGBM KDE

to_boxplot = pd.concat([highly_pred_microbes_ridge, highly_pred_microbes_lgbm] , axis=1).dropna()
to_boxplot.columns = ["Ridge", "LGBM"]
# to_boxplot.boxplot(flierprops=dict(markersize=2))
to_boxplot.plot.kde()
plt.title("Pearson Score distributions top correlations Ridge VS LGBM")
# plt.ylabel("Pearson Corr")
plt.xticks(rotation=45)
plt.grid()
plt.savefig(home_path + "figures/reverse/ridge_vs_lgbm_dist_top_corr_kde.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

In [ ]:
# Top prediction Ridge VS LightGBM boxplot
to_boxplot = pd.concat([highly_pred_microbes_ridge, highly_pred_microbes_lgbm] , axis=1).dropna()
U, p = stats.mannwhitneyu(highly_pred_microbes_ridge, highly_pred_microbes_lgbm, alternative='two-sided')
to_boxplot.columns = ["Ridge", "LightGBM"]
to_boxplot.boxplot(flierprops=dict(markersize=2))
plt.title("Pearson Corr distributions of top correlations Ridge VS LightGBM")
plt.ylabel("Pearson Corr")
# plt.xticks(rotation=45)
plt.grid()
plt.text(1.5, plt.ylim()[0] + 0.05 * (plt.ylim()[1] - plt.ylim()[0]), f'p-value: {p:.2f}', ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
plt.savefig(home_path + "figures/reverse/ridge_vs_lgbm_dist_top_corr_box.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
plt.show()
plt.clf()

#### Robustness Check

Abundance:

In [ ]:
def read_robustness(TARGETS):

    MODEL = 'LGBM' # 'LGBM' or 'ridge'
    STAGE = 'baseline' # 'baseline' or '02_visit'
    # suffix = '_all_baseline' if SPLIT == 'all_baseline' else '' 
    PROBLEM = '' # 'classification' or '' for regression
    robustness_results = []

    test_02_visit = pd.read_pickle("/home/tomerse/PycharmProjects/pythonProject/data/diet_mb_02_visit.pkl")
        
    baseline_subjects = list(diet_mb.index)
    visit_subjects = list(test_02_visit.index)
    print("Length of baseline_subjects:", len(baseline_subjects))
    print("Length of visit_subjects:", len(visit_subjects))
    test_baseline = diet_mb.loc[diet_mb.index.isin(visit_subjects)]
    test_02_visit = test_02_visit.loc[test_02_visit.index.isin(test_baseline.index)]
    print(test_baseline.shape)  
    print(test_02_visit.shape) 
    # test_baseline is my holdout set 
    current_df = diet_mb.loc[~diet_mb.index.isin(visit_subjects)].copy()

    n_samples = len(current_df)
    samples = []

    iteration = 1
    while n_samples > 1:
        samples.append(n_samples)
        print(f"Iteration {iteration}: {n_samples} samples")
        loaded = pickle.load(open("/net/mraid20/export/genie/LabData/Analyses/tomerse/test/reverse/robustness/output_" + MODEL + '_' + TARGETS + '_' + str(n_samples) + '_samples' + PROBLEM + '.pkl', "rb"))
        robustness_results.append(loaded)
        # Reduce samples by half for the next iteration
        n_samples = math.ceil(n_samples / 2)  # Use math.ceil to ensure at least one sample
        current_df = current_df.sample(n=n_samples, random_state=iteration).reset_index(drop=True)
        iteration += 1

    return robustness_results, samples


def robustness_calc(robustness_results, samples, TARGETS):
    if TARGETS == "div":
        print("div")
        robustness_scores_richness = []
        robustness_scores_shannon = []
        for result in robustness_results:
            robustness_scores, robustness_pvalues, robustness_coefs, robustness_preds, robustness_targets = read_results(result)
            robustness_scores_richness.append(robustness_scores[0])
            robustness_scores_shannon.append(robustness_scores[1])

        nan_index = np.isnan(robustness_scores_richness).argmax()
        robustness_scores_richness = robustness_scores_richness[:nan_index]
        robustness_scores_shannon = robustness_scores_shannon[:nan_index]
        samples = samples[:nan_index]

        iterations = list(range(1, nan_index + 1))

        return robustness_scores_richness, robustness_scores_shannon, iterations, samples

    elif TARGETS == "abundance":
        print("abundance")
        robustness_means = []
        robustness_stds = []
        for result in robustness_results:
            robustness_scores, robustness_pvalues, robustness_coefs, robustness_preds, robustness_targets = read_results(result)
            robustness_means.append(robustness_scores.mean())
            robustness_stds.append(robustness_scores.std())


        nan_index = np.isnan(robustness_means).argmax()
        robustness_means = robustness_means[:nan_index]
        robustness_stds = robustness_stds[:nan_index]
        samples = samples[:nan_index]

        iterations = list(range(1, nan_index + 1))

        return robustness_means, robustness_stds, iterations, samples


In [ ]:
robustness_results, samples = read_robustness("abundance")
robustness_means, robustness_stds, iterations, samples = robustness_calc(robustness_results, samples, "abundance")

print(robustness_means)
print(robustness_stds)
print(iterations)
print(samples)

In [ ]:
# Abundance

def plot_robustness_abundance(robustness_means, robustness_stds, samples):
    # Plot mean with standard deviations as error bars
    plt.figure(figsize=(12, 6))
    plt.errorbar(
        samples,
        robustness_means,
        yerr=robustness_stds,
        fmt='-o',
        capsize=5,
        label="Mean Pearson Correlations",
        # ecolor='black', 
        color='cadetblue'
    )

    # Formatting
    # plt.title("Mean Pearson Correlations per Number of Subjects")
    plt.xlabel("Number of Subjects per Iteration", fontsize=14)
    plt.ylabel("Mean Pearson Correlation ± STD", fontsize=14)
    # plt.xscale("log")  # Log scale for better visibility
    plt.xticks(samples, rotation=90)
    plt.grid(alpha=0.5, linestyle='--')
    # plt.legend()

    # Show plot
    plt.tight_layout()
    plt.savefig(home_path + "figures/reverse/abundance_robustness.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
    plt.show()


plot_robustness_abundance(robustness_means, robustness_stds, samples)

Diversity:

In [ ]:
robustness_results, samples = read_robustness("div")
robustness_scores_richness, robustness_scores_shannon, iterations, samples = robustness_calc(robustness_results, samples, "div")

print(robustness_scores_richness)
print(robustness_scores_shannon)
print(iterations)
print(samples)

In [ ]:
# Abundance

def plot_robustness_div(robustness_scores_richness, robustness_scores_shannon, samples):
    plt.figure(figsize=(12, 6))
    
    # Plot the two lines
    plt.plot(samples, robustness_scores_richness, label="Richness", marker='o', linestyle='-', linewidth=2, color='cadetblue')
    plt.plot(samples, robustness_scores_shannon, label="Shannon Index", marker='o', linestyle='-', linewidth=2, color='coral')
    
    # Formatting
    # plt.title("Diversity Pearson's Correlations per Number of Subjects")
    plt.xlabel("Number of Subjects per Iteration", fontsize=14)
    plt.ylabel("Pearson Correlation", fontsize=14)
    plt.xticks(samples, rotation=90)
    plt.grid(alpha=0.5, linestyle='--')
    plt.legend()

    # Show plot
    plt.tight_layout()
    plt.savefig(home_path + "figures/reverse/diversity_robustness.png", dpi=300, facecolor="white", transparent=False, bbox_inches='tight')
    plt.show()
    

plot_robustness_div(robustness_scores_richness, robustness_scores_shannon, samples)